# Deber 3 - P3: Mejora analítica de mantenimientos

Complete las celdas marcadas con `TODO`. La solución debe usar archivos, pandas,
NumPy, matplotlib, ciclos, condicionales, validaciones y manejo de errores.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Carga de datos y manejo de errores

In [ ]:
def cargar_datos(ruta: str) -> pd.DataFrame | None:
    # TODO:
    # 1. Leer el CSV con pandas.
    # 2. Manejar FileNotFoundError.
    # 3. Manejar archivo vacío.
    # 4. Manejar error de formato CSV.
    # 5. Retornar DataFrame o None.
    pass
def cargar_datos(ruta: str) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(ruta)

        if df.empty:
            print("Error: el archivo está vacío.")
            return None

        print(f"Archivo cargado correctamente: {ruta}")
        print(f"Registros encontrados: {len(df)}")

        return df

    except FileNotFoundError:
        print(f"Error: no se encontró el archivo '{ruta}'.")
        return None

    except pd.errors.EmptyDataError:
        print("Error: el archivo CSV está vacío.")
        return None

    except pd.errors.ParserError:
        print("Error: el formato del archivo CSV no se puede interpretar.")
        return None

    except Exception as e:
        print(f"Error inesperado al cargar el archivo: {e}")
        return None

In [ ]:
# PRUEBA 1: archivo válido
df = cargar_datos("mantenimientos.csv")

In [ ]:
# PRUEBA 2: archivo inexistente
_ = cargar_datos("archivo_que_no_existe.csv")

## 2. Validación y limpieza

In [ ]:
EQUIPOS_VALIDOS = {
    "Laptop", "Desktop", "Impresora", "Proyector", "Router", "Switch"
}
ESTADOS_VALIDOS = {"Completado", "Pendiente", "En proceso"}


def validar_y_limpiar(df: pd.DataFrame):
    # TODO:
    # Detectar al menos:
    # - código vacío
    # - código duplicado
    # - fecha inválida
    # - costo no numérico o negativo
    # - duración no numérica o <= 0
    # - satisfacción fuera de 1 a 5
    # - estado no permitido
    # - equipo no permitido
    # - área vacía
    #
    # Debe existir al menos un ciclo con finalidad real.
    pass
def validar_y_limpiar(df: pd.DataFrame):

    df = df.copy()
    errores = []

    # Convertir columnas numéricas
    df["costo"] = pd.to_numeric(df["costo"], errors="coerce")
    df["duracion"] = pd.to_numeric(df["duracion"], errors="coerce")
    df["satisfaccion"] = pd.to_numeric(df["satisfaccion"], errors="coerce")

    # Convertir fecha
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    # Reglas de validación
    reglas = {
        "código vacío": df["codigo"].isna() | (df["codigo"].astype(str).str.strip() == ""),
        "código duplicado": df["codigo"].duplicated(keep=False),
        "fecha inválida": df["fecha"].isna(),
        "costo no válido": df["costo"].isna() | (df["costo"] < 0),
        "duración no válida": df["duracion"].isna() | (df["duracion"] <= 0),
        "satisfacción fuera de rango": df["satisfaccion"].isna() | ~df["satisfaccion"].between(1, 5),
        "estado no permitido": ~df["estado"].isin(ESTADOS_VALIDOS),
        "equipo no permitido": ~df["equipo"].isin(EQUIPOS_VALIDOS),
        "área vacía": df["area"].isna() | (df["area"].astype(str).str.strip() == "")
    }

    # Ciclo para recorrer todas las reglas
    for nombre, condicion in reglas.items():
        cantidad = condicion.sum()

        if cantidad > 0:
            errores.append(f"{nombre}: {cantidad} registro(s)")

    # Mostrar errores encontrados
    print("===== ERRORES DETECTADOS =====")

    if errores:
        for error in errores:
            print("-", error)
    else:
        print("No se encontraron errores.")

    # Eliminar registros inválidos
    mascara_valida = (
        ~reglas["código vacío"] &
        ~reglas["código duplicado"] &
        ~reglas["fecha inválida"] &
        ~reglas["costo no válido"] &
        ~reglas["duración no válida"] &
        ~reglas["satisfacción fuera de rango"] &
        ~reglas["estado no permitido"] &
        ~reglas["equipo no permitido"] &
        ~reglas["área vacía"]
    )

    df_limpio = df[mascara_valida].copy()

    print(f"\nRegistros originales: {len(df)}")
    print(f"Registros válidos: {len(df_limpio)}")
    print(f"Registros excluidos: {len(df) - len(df_limpio)}")

    return df_limpio

In [ ]:
# PRUEBA 3: archivo con errores deliberados
df_error = cargar_datos("mantenimientos_con_errores.csv")

if df_error is not None:
    df_error_limpio = validar_y_limpiar(df_error)

    print("\n===== DATAFRAME LIMPIO =====")
    display(df_error_limpio)

## 3. Análisis con NumPy

In [ ]:
ostos = df_limpio["costo"].dropna().to_numpy()

media = np.mean(costos)
mediana = np.median(costos)
percentil_75 = np.percentile(costos, 75)

clasificacion = np.where(
    costos >= media,
    "Alto",
    "Bajo"
)

print("===== ANÁLISIS CON NUMPY =====")
print(f"Media de costo: ${media:.2f}")
print(f"Mediana de costo: ${mediana:.2f}")
print(f"Percentil 75: ${percentil_75:.2f}")
print("Clasificación de costos:", clasificacion)

## 4. Análisis con pandas

In [ ]:
# 4. ANÁLISIS CON PANDAS

print("===== ANÁLISIS CON PANDAS =====")

# Número de registros válidos
num_validos = len(df_limpio)

# Costo total
costo_total = df_limpio["costo"].sum()

# Costo promedio
costo_promedio = df_limpio["costo"].mean()

# Mantenimiento más costoso
mantenimiento_costoso = df_limpio.loc[df_limpio["costo"].idxmax()]

# Cantidad de mantenimientos por área
cantidad_por_area = df_limpio.groupby("area").size()

# Costo promedio por tipo de mantenimiento
costo_promedio_tipo = df_limpio.groupby("tipo_mantenimiento")["costo"].mean()

print(f"Número de registros válidos: {num_validos}")
print(f"Costo total: ${costo_total:.2f}")
print(f"Costo promedio: ${costo_promedio:.2f}")

print("\nMantenimiento más costoso:")
print(mantenimiento_costoso)

print("\nCantidad de mantenimientos por área:")
print(cantidad_por_area)

print("\nCosto promedio por tipo de mantenimiento:")
print(costo_promedio_tipo)

## 5. Visualización con matplotlib

In [ ]:
# GRÁFICO 1: Costo total de mantenimientos por área

costo_por_area = df_limpio.groupby("area")["costo"].sum()

plt.figure(figsize=(8, 5))
plt.bar(costo_por_area.index, costo_por_area.values)

plt.title("Costo total de mantenimientos por área")
plt.xlabel("Área")
plt.ylabel("Costo total ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show().

In [ ]:
# GRÁFICO 2: Cantidad de mantenimientos por tipo

cantidad_por_tipo = df_limpio.groupby("tipo_mantenimiento").size()

plt.figure(figsize=(8, 5))
plt.bar(cantidad_por_tipo.index, cantidad_por_tipo.values)

plt.title("Cantidad de mantenimientos por tipo")
plt.xlabel("Tipo de mantenimiento")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Exportación

In [ ]:
# 6. EXPORTACIÓN

# Guardar los datos limpios
df_limpio.to_csv("mantenimientos_limpios.csv", index=False)

# Crear resumen por área
resumen_por_area = df_limpio.groupby("area").agg(
    cantidad_mantenimientos=("codigo", "count"),
    costo_total=("costo", "sum"),
    costo_promedio=("costo", "mean")
).reset_index()

# Guardar resumen
resumen_por_area.to_csv("resumen_por_area.csv", index=False)

print("===== ARCHIVOS EXPORTADOS =====")
print("✓ mantenimientos_limpios.csv")
print("✓ resumen_por_area.csv")

print("\n===== RESUMEN POR ÁREA =====")
display(resumen_por_area)

## 7. Conclusión

 El algoritmo carga los registros de mantenimiento desde archivos CSV, verifica la calidad de los datos, identifica y excluye registros inválidos, y posteriormente analiza la información utilizando Pandas y NumPy. Finalmente, genera gráficos y exporta los resultados en archivos CSV.

Se utilizó un ciclo for para recorrer las diferentes reglas de validación y detectar los errores de cada registro. También se utilizaron estructuras if/else para controlar las decisiones del programa.

Las validaciones realizadas incluyen códigos vacíos o duplicados, fechas inválidas, costos negativos o no numéricos, duraciones incorrectas, satisfacción fuera del rango de 1 a 5, estados o equipos no permitidos y áreas vacías.

El programa controla errores como archivos inexistentes, archivos vacíos y problemas de lectura del formato CSV mediante try/except.

Los resultados más importantes son el número de mantenimientos válidos, el costo total y promedio, el mantenimiento más costoso, los costos por área y la cantidad de mantenimientos por tipo.